# Lab W2: Baseline CNN pada CIFAR-10

## Pre-flight Checklist

> [!IMPORTANT]
> Baca cell ini sebelum eksekusi cell apapun. Konsep yang ditandai (§) merujuk ke `02_W2_Images_CNN_Smoke_Test.md`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W2 sudah dibaca, terutama §1.5 (citra→tensor 4D), §2.3 (smoke test 3-level), §2.5 (cara kerja Conv2d), §2.7 (Kaiming init + BatchNorm), §2.8 (augmentation/dropout/regularization).
- Lab W1 (`lab_w1_tabular_heads.ipynb`) sudah selesai (Anda sudah pernah training PyTorch pipeline).
- Familiar dengan tuple shape `(B, C, H, W)` (lihat §1.5.3 W2).
- `pip install -e .` dari `template_repo/` sukses; CIFAR-10 akan auto-download di run pertama (~170 MB).

**Yang akan Anda hasilkan di akhir lab:**
- Smoke test 3-level berhasil (import, dummy forward, overfit satu batch).
- Pipeline dry-run lolos (integrasi config, dataloader, training loop 1 epoch).
- Baseline `SimpleCNN` terlatih 30 epoch pada CIFAR-10; `val_acc > 0.70`.
- Baseline `ResNet-18` pretrained frozen backbone; `val_acc > 0.85`.
- Loss/accuracy curve diplot dari log; gap train-val di-flag.
- Confusion matrix test set + per-class accuracy.
- 10 prediksi confident-salah dianalisis untuk pola.
- Dokumentasi: di level mana smoke test menangkap setiap jenis error.
- Tiga refleksi tertarget.

**Resource:**
- **Hardware:** GPU **sangat dianjurkan** untuk training penuh (5 menit GPU vs 15-25 menit CPU). Smoke test cukup CPU.
- **Storage:** ~250 MB (CIFAR-10 dataset + checkpoint).
- **Estimasi waktu kerja:** 3-5 jam termasuk training, analisis, dan refleksi.

**Pendamping:** Bab W2 di `02_W2_Images_CNN_Smoke_Test.md`.

## Tujuan
1. Memverifikasi template repo dapat dijalankan end-to-end (smoke test 3-level lolos).
2. Melatih baseline `SimpleCNN` pada CIFAR-10 selama 30 epoch.
3. Melatih baseline `ResNet-18` pretrained dengan frozen backbone.
4. Memplot kurva loss dan akurasi per epoch dari log.
5. Menghitung confusion matrix pada test set.
6. Menganalisis 10 prediksi paling salah-yakin.

## Daftar Periksa Eksekusi
- [ ] Smoke test Level 1 (import + arsitektur print) di §1.
- [ ] Smoke test Level 2 (dummy forward shape match) di §1.
- [ ] Smoke test Level 3 (overfit satu batch, loss < 0.1) di §1.
- [ ] Pipeline dry-run (1 epoch integrasi) di §2.
- [ ] Training penuh (30 epoch) selesai, val_acc > 0.70 di §3.
- [ ] Plot kurva loss/accuracy + flag gap di §4.
- [ ] Confusion matrix test set + per-class accuracy di §5.
- [ ] 10 prediksi confident-salah dianalisis di §6.
- [ ] ResNet-18 pretrained frozen backbone terlatih di §7.
- [ ] Dokumentasi error-level smoke test di §8.
- [ ] Refleksi tiga pertanyaan di §9.

## 0. Setup

Jalankan dari folder `template_repo/`. Pastikan environment sudah aktif:
```bash
pip install -e .
```

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
# Di Google Colab: clone repo terlebih dahulu, lalu cd ke template_repo/
#   git clone <url-repo> && cd template_repo && pip install -e .
# Di environment lokal: jalankan dari folder notebooks/
import sys, os
_root = os.path.abspath('..')
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Root:", _root)

In [ ]:
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Smoke Test Level 1 dan Level 2

Cell di bawah menjalankan dua level smoke test pertama dari §2.3 W2:

- **Level 1 (Import test):** `from src.models import SimpleCNN; model = SimpleCNN()`. Kalau gagal di sini, ada typo, dependency hilang, atau shape mismatch di definisi layer.
- **Level 2 (Dummy forward):** `out = model(torch.randn(4, 3, 32, 32))` lalu verifikasi `out.shape == (4, 10)`. Kalau gagal di sini, ada bug shape antar layer (bukan masalah data).

**Apa isi `SimpleCNN`?** Dari `src/models.py`: dua blok konvolusi, masing-masing `Conv2d → BatchNorm2d → ReLU → Conv2d → BatchNorm2d → ReLU → MaxPool2d` — total 4 layer Conv. Progresi channel: 3 → 32 → 64 → 128 → 128. Output shape setelah blok 2: `(B, 128, 8, 8)` (resolusi 32→16→8 setelah dua pooling). Classifier: `AdaptiveAvgPool2d(1) → Flatten → Dropout(0.3) → Linear(128, 10)`. Untuk derivasi shape, lihat rumus output shape di §2.5.3 W2.

In [ ]:
from src.models import SimpleCNN
from src.utils import count_parameters

model = SimpleCNN(num_classes=10)
print(model)
print(f'\ntrainable params: {count_parameters(model):,}')

# Verifikasi forward pass - input CIFAR-10: (B, 3, 32, 32)
dummy = torch.randn(4, 3, 32, 32)
out = model(dummy)
print(f'input shape: {dummy.shape} → output shape: {out.shape}')  # harus (4, 10)

## 1b. Smoke Test Level 3 — Overfit Satu Batch

Level 3 dari §2.3 W2: buktikan bahwa gradient mengalir dari loss ke seluruh parameter model. Strateginya: ambil 8 sampel CIFAR-10, training 100 iterasi pada batch yang sama, pastikan loss turun dari ≈2.3 ke <0.1.

Ini adalah tes diagnostik paling informatif. Kalau gagal di sini, semua training berikutnya sia-sia.

In [ ]:
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)

model_overfit = SimpleCNN(num_classes=10).to(device)
model_overfit.train()
optimizer = torch.optim.AdamW(model_overfit.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Ambil 8 sampel — cukup untuk overfit tanpa augmentasi
small_loader = DataLoader(Subset(dataset, range(8)), batch_size=8, shuffle=True)
x, y = next(iter(small_loader))
x, y = x.to(device), y.to(device)

losses = []
for i in range(100):
    optimizer.zero_grad()
    loss = criterion(model_overfit(x), y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if i % 20 == 0:
        print(f'iter {i:3d}: loss = {loss.item():.4f}')

final_loss = losses[-1]
print(f'\nLoss awal: {losses[0]:.4f} → Loss akhir: {final_loss:.4f}')
if final_loss < 0.1:
    print('✓ Level 3 smoke test: overfit satu batch berhasil (loss < 0.1). Gradient mengalir.')
elif final_loss < 0.5:
    print(f'⚠ Loss {final_loss:.3f} < 0.5, turun tapi belum cukup. Coba tambah iterasi atau cek lr.')
else:
    print(f'✗ Loss masih {final_loss:.3f}. Kemungkinan: model dalam eval mode, lr terlalu kecil, atau bug di arsitektur.')

## 2. Pipeline Dry-Run

Setelah 3-level smoke test lolos, jalankan pipeline integrasi: config YAML + dataloader + training loop 1 epoch dengan data sintetis kecil. Ini memverifikasi seluruh `src/train.py` dapat dijalankan tanpa error sebelum kita investasi waktu di training penuh.

In [ ]:
import subprocess, sys, os
result = subprocess.run(
    [sys.executable, '-m', 'src.train', '--config', 'configs/baseline.yaml', '--dry-run'],
    capture_output=True, text=True,
    cwd=os.path.abspath('..')
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
assert result.returncode == 0, 'Dry-run gagal! Periksa error di atas.'

## 3. Training Penuh

Training lebih cepat dari terminal daripada dalam notebook. Jalankan:
```bash
python -m src.train --config configs/baseline.yaml --seed 42
```
Tunggu selesai (≈10-20 menit CPU, <5 menit GPU), lalu lanjutkan sel di bawah.

## 4. Baca Hasil dan Plot Kurva Training

In [ ]:
import json
from pathlib import Path

exp_dir = Path('../experiments/baseline_seed42')
summary = json.loads((exp_dir / 'summary.json').read_text())
print('Summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Baca metrics dari log (parse train.log untuk per-epoch data)
import re

log_path = exp_dir / 'train.log'
epochs_data = []
pattern = re.compile(
    r'epoch=(\d+) train_loss=([\d.]+) train_acc=([\d.]+) val_loss=([\d.]+) val_acc=([\d.]+)'
)
for line in log_path.read_text(encoding='utf-8').splitlines():
    m = pattern.search(line)
    if m:
        epochs_data.append({
            'epoch': int(m.group(1)),
            'train_loss': float(m.group(2)),
            'train_acc': float(m.group(3)),
            'val_loss': float(m.group(4)),
            'val_acc': float(m.group(5)),
        })

df_log = pd.DataFrame(epochs_data)
print(f'Epoch terbaca: {len(df_log)}')
df_log.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot loss
axes[0].plot(df_log['epoch'], df_log['train_loss'], label='train', color='steelblue')
axes[0].plot(df_log['epoch'], df_log['val_loss'], label='val', color='tomato')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss per Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot akurasi
axes[1].plot(df_log['epoch'], df_log['train_acc'] * 100, label='train', color='steelblue')
axes[1].plot(df_log['epoch'], df_log['val_acc'] * 100, label='val', color='tomato')
axes[1].axhline(70, color='gray', linestyle='--', alpha=0.5, label='target 70%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy per Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(exp_dir / 'loss_acc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

# Deteksi overfitting
last_row = df_log.iloc[-1]
gap = last_row['train_acc'] - last_row['val_acc']
print(f'\nGap train-val akurasi di epoch terakhir: {gap*100:.1f}%')
if gap > 0.15:
    print('⚠ Gap > 15%  -  indikasi overfitting. Coba: augmentasi lebih kuat, weight decay, dropout.')
elif gap < 0.02:
    print('⚠ Gap < 2%  -  mungkin underfitting atau dataset terlalu mudah.')
else:
    print('✓ Gap wajar.')

**Apa yang Anda perhatikan di kurva loss/accuracy?**

- **Bentuk kurva train.** Apakah loss turun mulus, atau ada plateau / lonjakan? Plateau panjang di awal = mungkin lr terlalu kecil; lonjakan tajam = lr terlalu besar atau gradient explode (cek di W3-W4).
- **Bentuk kurva val.** Apakah val mengikuti train, atau val datar sementara train terus turun? Pola "train turun, val datar" = overfitting klasik, dibahas di W3 §2.5 (diagnosis loss curve).
- **Gap akhir.** Gap < 0.02 = mungkin underfitting (model belum cukup ekspresif atau training belum cukup epoch); gap 0.02-0.10 = berjalan baik; gap > 0.15 = overfitting, butuh regularisasi lebih kuat (augmentation, weight decay, dropout - lihat §2.8 W2).
- **Akurasi target.** Untuk SimpleCNN baseline di CIFAR-10 dengan augmentasi standar dan AdamW, val_acc target ≈ 75-80% di epoch 30. Kalau Anda jauh di bawah ini, coba cek BatchNorm aktif di training (`model.train()`), augmentasi tidak diterapkan ke val/test, dan lr scheduler tidak terlalu agresif.

---

In [ ]:
import numpy as np
from src.models import build_model
from src.utils import load_config
from src.data import build_loaders

cfg = load_config('../configs/baseline.yaml')

model_eval = build_model(cfg['model']).to(device)
ckpt = torch.load(exp_dir / 'ckpt_best.pt', map_location=device)
model_eval.load_state_dict(ckpt['model_state'])
model_eval.eval()
print('Loaded checkpoint - epoch:', ckpt['meta']['epoch'], '| val_acc:', ckpt['meta']['metric_value'])

In [ ]:
_, _, test_loader = build_loaders(cfg['data'])

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = torch.as_tensor(y).long().view(-1)
        logits = model_eval(x)
        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_acc = (all_preds == all_labels).mean()
print(f'Test accuracy: {test_acc*100:.2f}%')

**Apa yang Anda perhatikan di confusion matrix?**

- **Diagonal kuat?** Kelas dengan diagonal > 0.80 = mudah dikenali; < 0.60 = sulit. Di CIFAR-10 baseline, biasanya `cat`, `dog`, `bird` paling sulit; `automobile`, `truck`, `ship` paling mudah.
- **Pasangan bingung simetris?** Kalau cm[cat, dog] tinggi DAN cm[dog, cat] tinggi, dua kelas saling bingung (semantically dekat). Kalau hanya satu arah tinggi, ada bias prediksi (model lebih sering predict A daripada B).
- **Per-class accuracy yang rendah.** Kelas dengan acc < 0.60 sering punya satu dari ini: (a) jumlah sampel sedikit, (b) variasi visual tinggi (cat punya ras, postur, angle yang sangat beragam), (c) overlap fitur dengan kelas lain.

---

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=CLASSES, yticklabels=CLASSES,
    vmin=0, vmax=1, ax=ax
)
ax.set_xlabel('Prediksi')
ax.set_ylabel('Label Asli')
ax.set_title('Confusion Matrix (normalized) - Test Set')
plt.tight_layout()
plt.savefig(exp_dir / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# Cetak per-class accuracy
print('\nPer-class accuracy:')
for i, cls in enumerate(CLASSES):
    print(f'  {cls:12s}: {cm_norm[i, i]*100:.1f}%')

## 6. Analisis Kesalahan: 10 Prediksi Paling Confident-Salah

Ini adalah langkah yang sering dilewati tapi sangat informatif: di mana model *paling yakin tapi salah*?

**Apa yang Anda perhatikan di gambar confident-salah?**

Inspeksi visual setiap gambar dan tanyakan:

- **Apakah ground truth-nya jelas?** Beberapa gambar CIFAR-10 ambigu bahkan untuk manusia (bird ratio kecil di langit, cat dari belakang yang terlihat seperti dog). Confident-salah pada gambar ambigu = wajar, model berperilaku rasional.
- **Pola pencahayaan / angle / oklusi?** Kalau Anda lihat 5+ gambar punya pencahayaan ekstrem atau angle non-frontal, augmentasi training mungkin belum cukup mencakup variasi ini. W3 akan membahas strategi augmentasi.
- **Texture vs shape bias.** Model CNN sering bias ke texture daripada shape (Geirhos et al., 2019, arXiv:1811.12231). Cat berbulu lurus diprediksi `dog` karena texture mirip. Kalau pola ini muncul berulang, ada problem yang dibahas di W6-W7 (representation bias).
- **Alasan model "yakin".** Walaupun salah, prob > 0.9 berarti model sangat percaya. Apa yang membuatnya yakin? Bayangkan filter awal CNN melihat tepi/warna lokal; mungkin gambar punya tepi atau warna yang strongly cocok kelas yang salah.

---

In [ ]:
# Confidence = probabilitas kelas yang diprediksi
pred_confidences = all_probs[np.arange(len(all_preds)), all_preds]

# Filter hanya yang salah
wrong_mask = all_preds != all_labels
wrong_indices = np.where(wrong_mask)[0]
wrong_confidences = pred_confidences[wrong_mask]

# Top-10 paling confident di antara yang salah
top10_idx = wrong_indices[np.argsort(wrong_confidences)[::-1][:10]]

print(f'Total prediksi salah: {wrong_mask.sum()} dari {len(all_labels)}')
print(f'\nTop-10 paling confident-salah:')
print(f'{"Idx":>6} {"True":>12} {"Pred":>12} {"Confidence":>12}')
for i in top10_idx:
    print(f'{i:>6} {CLASSES[all_labels[i]]:>12} {CLASSES[all_preds[i]]:>12} {pred_confidences[i]:>12.3f}')

## 7. Baseline ResNet-18 Pretrained (Frozen Backbone)

Module W2 §5 mewajibkan baseline kedua: model pretrained dengan backbone di-freeze, hanya head yang dilatih. Ini menguji transfer learning — apakah fitur dari ImageNet cukup untuk CIFAR-10 tanpa fine-tuning?

In [ ]:
import torchvision.models as tv_models

print('Memuat ResNet-18 pretrained dari ImageNet...')
resnet = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze semua parameter backbone
for p in resnet.parameters():
    p.requires_grad = False

# Ganti fc layer untuk CIFAR-10 (10 kelas), pakai head 2-layer
resnet.fc = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 10),
)
resnet = resnet.to(device)

# Smoke test: forward pass
dummy = torch.randn(2, 3, 32, 32).to(device)
with torch.no_grad():
    out = resnet(dummy)
print(f'Smoke test ResNet-18: input {dummy.shape} → output {out.shape}  # harus (2, 10)')

# Hitung parameter
trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in resnet.parameters())
print(f'Parameter: {trainable:,} trainable / {total:,} total  (backbone frozen)')

In [ ]:
# Training ResNet-18 dari terminal (backbone frozen, head dilatih dari nol)
# Gunakan lr lebih tinggi karena hanya head yang dilatih
print('Jalankan perintah berikut di terminal:')
print('  python -m src.train --config configs/baseline.yaml --seed 43')
print('  (Ganti nama model di config ke resnet18_frozen — atau edit configs/resnet_frozen.yaml)')
print()
print('Kalau mau langsung training inline:')
import subprocess, sys, os
print('Menjalankan training ResNet-18 frozen (seed 43)...')
# Training ResNet-18 pakai config terpisah atau override manual
# Untuk sekarang, catat bahwa val_acc target > 0.85 (transfer learning,
# backbone ImageNet sudah punya fitur sangat kuat untuk CIFAR-10).

### Analisis Perbandingan SimpleCNN vs ResNet-18 Frozen

Setelah kedua model terlatih, bandingkan:

| Aspek | SimpleCNN (scratch) | ResNet-18 (frozen) |
| --- | --- | --- |
| Parameter trainable | ~200K | ~400K (head only) |
| Epoch ke konvergen | ~20-25 | ~5-10 |
| val_acc target | >0.70 | >0.85 |
| Gap train-val | lebih besar (overfitting) | lebih kecil (backbone tetap) |

> **Pertanyaan analisis:** Mengapa ResNet-18 frozen mencapai akurasi lebih tinggi dengan epoch lebih sedikit, padahal parameter trainable-nya lebih sedikit? Konsep *representation transfer* dari pretraining akan dibahas mendalam di W7 §1.1.

---

## 8. Dokumentasi: Di Level Mana Smoke Test Menangkap Error?

Module W2 §5 meminta Anda mendokumentasikan secara spesifik: untuk setiap jenis error, di level smoke test mana error tersebut akan terdeteksi. Isi tabel di bawah:

| Jenis Error | Smoke Test Level | Kenapa Tertangkap di Sini? |
| --- | --- | --- |
| Typo nama layer (`Conv2d` → `Conv2D`) | | |
| Shape mismatch antar layer (in_channel beda) | | |
| Dimensi output classifier tidak cocok `num_classes` | | |
| Gradient tidak mengalir (mis. `detach()` tidak sengaja) | | |
| Data loader path salah / file tidak ditemukan | | |
| Augmentasi diterapkan ke validation set | | |
| `BatchNorm` tidak update statistik (model selalu `eval()`) | | |
| Checkpoint tidak menyimpan config/metadata | | |

Untuk setiap jenis error, sebutkan level spesifik (1, 2, 3, atau dry-run) dan alasan satu kalimat kenapa level tersebut akan menangkapnya.

## 9. Inspeksi Checkpoint Metadata

Checkpoint yang disimpan di `experiments/` menyertakan metadata eksperimen. Periksa isinya.

In [ ]:
print('Checkpoint metadata:')
for k, v in ckpt['meta'].items():
    print(f'  {k}: {v}')

print('\nConfig dalam checkpoint (sebagian):')
for section in ['model', 'loss', 'optim', 'scheduler']:
    print(f'  {section}: {ckpt["config"].get(section, {})}')

## 10. Refleksi

Jawab tiga pertanyaan di bawah. Tidak ada jawaban benar/salah; yang penting alasan.

1. **Pasangan kelas bingung.** Kelas mana yang paling sering dikacaukan satu sama lain (lihat confusion matrix)? Secara visual, kenapa masuk akal? Apa property visual yang membuat keduanya sulit dibedakan?

2. **Pola confident-salah.** Di antara 10 gambar confident-salah, sebutkan satu pola yang Anda perhatikan (pencahayaan, angle, oklusi, texture bias). Dari sudut pandang arsitektur model, apa yang membuat prediksi salah itu tampak meyakinkan?

3. **Keputusan desain SimpleCNN.** Pilih satu dari berikut yang sekarang paling Anda pahami **dengan alasan teknis** (bukan hanya "karena di paper begini"):
   - (a) Kenapa `padding=1` di setiap Conv (lihat §2.5.3 W2).
   - (b) Kenapa BatchNorm SEBELUM ReLU, bukan sesudah.
   - (c) Kenapa `Dropout(0.3)` hanya di classifier `Linear(128, 10)` tetapi tidak di antara Conv layer.
   - (d) Kenapa classifier akhir tidak punya `Softmax` (lihat §2.2.3 W1 + §2.2.4 W1).

### Self-Check Quick

- [ ] Smoke test 3-level lulus (Level 3: loss < 0.1).
- [ ] Pipeline dry-run lolos.
- [ ] Training 30 epoch selesai; `val_acc > 0.70` (target 0.75-0.80).
- [ ] ResNet-18 frozen baseline terlatih; `val_acc > 0.85`.
- [ ] Tabel dokumentasi error-level smoke test terisi.
- [ ] Gap train-val di-flag dengan benar (overfit / sehat / underfit).
- [ ] Confusion matrix + per-class accuracy diinterpretasi (bukan hanya divisualkan).
- [ ] 10 confident-salah dianalisis pola, bukan hanya dilihat.
- [ ] Saya bisa menjelaskan rumus output shape `out = (in - k + 2p)/s + 1` dan menerapkan ke arsitektur SimpleCNN.

### Jawaban Refleksi

**1. Kelas yang sering dikacaukan:**

> *[tulis di sini]*

**2. Pola di gambar confident-salah:**

> *[tulis di sini]*

**3. Keputusan desain yang dipahami:**

> *[tulis di sini]*